# Tidewatch on Microsoft Fabric
Scheduled notebook: pulls the Tidewatch package from GitHub, runs the refresh, lands the events table in the Lakehouse (Delta) and writes the rendered page to the Lakehouse Files area.

Attach a Lakehouse to this notebook before running. Schedule it under the notebook's Run settings (Monday and Friday 06:45 Atlantic/Bermuda).

In [ ]:
%pip install -q requests beautifulsoup4 lxml pyyaml ephem gitpython

In [ ]:
import os, subprocess, sys
REPO = 'https://github.com/mkatzsch/bermuda-hub.git'   # change to your repo
WORK = '/tmp/tidewatch'
if not os.path.exists(WORK):
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO, WORK])
else:
    subprocess.check_call(['git', '-C', WORK, 'pull'])
os.chdir(WORK)
sys.path.insert(0, WORK)

In [ ]:
# keep the SQLite database in the Lakehouse so history persists between runs
import shutil
LAKE_DB = '/lakehouse/default/Files/tidewatch/tidewatch.sqlite'
os.makedirs(os.path.dirname(LAKE_DB), exist_ok=True)
os.makedirs('data', exist_ok=True)
if os.path.exists(LAKE_DB):
    shutil.copy(LAKE_DB, 'data/tidewatch.sqlite')
from tidewatch.cli import main
if not os.path.exists(LAKE_DB):
    main(['seed'])
main(['run'])
shutil.copy('data/tidewatch.sqlite', LAKE_DB)
shutil.copy('site/index.html', '/lakehouse/default/Files/tidewatch/index.html')

In [ ]:
# land the events as a Delta table for Power BI
import sqlite3, pandas as pd
con = sqlite3.connect('data/tidewatch.sqlite')
df = pd.read_sql('SELECT view, year, m AS month, d AS day, cat, name, time, venue, cost, status, source, first_seen, last_seen, active FROM events', con)
df['event_date'] = pd.to_datetime(dict(year=df.year, month=df.month, day=df.day))
spark.createDataFrame(df).write.mode('overwrite').format('delta').saveAsTable('tidewatch_events')
print(len(df), 'rows written to tidewatch_events')